#### The 4 sets of features that need to be extracted are:

1. GLCM texture features (contrast, homogeneity etc) — captures how "rough" or "smooth" the tissue texture is. Useful for Infiltration/Effusion, which show up as diffuse density changes rather than sharp shapes.
2. HOG (edge/gradient) features — captures shapes and boundaries. Useful for Nodules (small round opacities with distinct edges) and Atelectasis.
3. Regional intensity statistics (mean, std, skew, kurtosis) split into upper/middle/lower lung thirds — useful for Effusion, since fluid pools at the base of the lungs due to gravity. Projected via numpy.
4. Left-right symmetry score — many conditions (like Pneumothorax) are one-sided, so comparing left lung stats vs. right lung stats directly captures that asymmetry.

#### What we're doing here, step by step:

1. Loading each segmented image (from Outputs/segmented_train/ and Outputs/segmented_test/)
2. Run all 4 feature-family functions on it
3. Combine all the numbers into one row per image
4. Repeat for every image, building one big table
5. Save table as a CSV — this becomes the actual input to the models next week

In [ ]:
#import required libraries
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops, hog
from scipy.stats import skew, kurtosis
import os
from glob import glob

In [ ]:
#Calculates Gray-Level Co-occurrence Matrix (GLCM) features to analyze texture. Measures Contrast,
#Homogeneity etc
def extract_glcm_features(img):
    glcm = graycomatrix(img, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    return {
        'glcm_contrast': graycoprops(glcm, 'contrast')[0, 0],
        'glcm_homogeneity': graycoprops(glcm, 'homogeneity')[0, 0],
        'glcm_energy': graycoprops(glcm, 'energy')[0, 0],
    }

In [ ]:
#Extracts Histogram of Oriented Gradients (HOG) features to capture structural shapes and edges.
#Computes edge orientations using 9 directional bins into two summary metrics:
# (i) mean and (ii) standard deviation.
def extract_hog_features(img):
    hog_features = hog(img, orientations=9, pixels_per_cell=(32, 32),
                        cells_per_block=(2, 2), feature_vector=True)
    return {
        'hog_mean': np.mean(hog_features),
        'hog_std': np.std(hog_features),
    }

In [ ]:
#Evaluates pixel intensity distribution across three horizontal zones (Upper, Middle, and Lower lungs).
#For each region, it extracts: Mean/SD, Skewness, Kurtosis
def extract_regional_stats(img):
    h = img.shape[0]
    upper = img[0:h//3, :]
    middle = img[h//3:2*h//3, :]
    lower = img[2*h//3:h, :]
    features = {}
    for name, region in [('upper', upper), ('middle', middle), ('lower', lower)]:
        vals = region.flatten()
        features[f'{name}_mean'] = np.mean(vals)
        features[f'{name}_std'] = np.std(vals)
        features[f'{name}_skew'] = skew(vals)
        features[f'{name}_kurtosis'] = kurtosis(vals)
    return features

In [ ]:
#Measures bilateral & anatomical symmetry between the left and right lung halves.
#Splits the image down the center, flips the right half horizontally, trims any odd-pixel alignment
#mistakes and calculates the Mean Absolute Difference (MAE) between both sides.
#Lower MAE values indicate higher left-right symmetry.
def extract_symmetry_score(img):
    w = img.shape[1]
    left = img[:, 0:w//2]
    right = img[:, w//2:w]
    right_flipped = np.fliplr(right)
    # resize in case of odd width mismatch
    min_w = min(left.shape[1], right_flipped.shape[1])
    left = left[:, :min_w]
    right_flipped = right_flipped[:, :min_w]
    diff = np.abs(left.astype(float) - right_flipped.astype(float))
    return {'symmetry_score': np.mean(diff)}

### Point - 1
#### Nodule is still the weakest link (0.556 AUROC). Nodules are small, round,sharp-edged opacities Current features (GLCM texture, symmetry, etc) aren't designed to detect small localized round shapes. 

#### Hence, we added a blob detection feature:

In [ ]:
from skimage.feature import blob_log
def extract_blob_features(img):
    blobs = blob_log(img, min_sigma=2, max_sigma=10, num_sigma=5, threshold=0.05)
    return {
        'num_blobs': len(blobs),
        'max_blob_size': blobs[:, 2].max() if len(blobs) > 0 else 0,
        'mean_blob_size': blobs[:, 2].mean() if len(blobs) > 0 else 0,
    }

In [6]:
#Reads the image off disk in grayscale and aggregates all texture, shape, regional, and
#symmetry metrics into a single dictionary alongside the filename.
def extract_all_features(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    features = {'filename': os.path.basename(img_path)}
    features.update(extract_glcm_features(img))
    features.update(extract_hog_features(img))
    features.update(extract_regional_stats(img))
    features.update(extract_symmetry_score(img))
    return features

In [7]:
#Searches a folder for all .png files, extracts features for each image sequentially,
#and compiles everything into a clean pandas.DataFrame where each row represents one image.
def build_feature_table(folder_path):
    image_paths = glob(os.path.join(folder_path, '*.png'))
    rows = []
    for path in image_paths:
        rows.append(extract_all_features(path))
    return pd.DataFrame(rows)

In [ ]:
train_features = build_feature_table('Outputs/segmented_train')

In [ ]:
test_features = build_feature_table('Outputs/segmented_test')

In [ ]:
#Features for each img loaded to train/test csv files.
train_features.to_csv('Outputs/train_features.csv', index=False)
test_features.to_csv('Outputs/test_features.csv', index=False)

In [ ]:
#Here we are storing the train data (png files) as csv files that can be easily stored.
#We can use the train data (in the form of csv) to train the model
labels = pd.read_csv('train_labels.csv')
merged = train_features.merge(labels, left_on='filename', right_on=labels.columns[0])
print(merged.shape)  # It equals train_features row count, as expected